# Vienna Orthofoto Test

This notebook downloads a small City of Vienna Orthofoto / ViennaGIS WMTS image around Vienna center and displays it.

An orthophoto is an aerial photograph that has been geometrically corrected so it lines up with map coordinates. For this Vienna-only project, it is a better primary source than normal satellite imagery.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

In [ ]:
# Find the project root so this notebook works from either the repo root or notebooks/.
current_dir = Path.cwd().resolve()

if (current_dir / "notebooks").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

# Add the project root to Python's import path so we can import backend helpers.
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

project_root

In [ ]:
from backend.app.imagery.vienna_orthofoto import (
    download_vienna_orthofoto,
    image_pixel_to_lonlat,
    lonlat_to_webmercator,
    webmercator_to_lonlat,
)

In [ ]:
# Vienna center test location.
# This request downloads an aerial orthophoto from ViennaGIS, not a generic satellite image.
lat = 48.22057422849521
lon = 16.411494407045154
zoom = 18
image_width = 512
image_height = 512

image_path = project_root / "data" / "vienna_orthofoto_test.png"
metadata_path = project_root / "data" / "vienna_orthofoto_test_metadata.json"

metadata = download_vienna_orthofoto(
    lat=lat,
    lon=lon,
    zoom=zoom,
    width=image_width,
    height=image_height,
    output_path=str(image_path),
)

# Save the georeferencing metadata next to the PNG so later tree/canopy models
# can convert image pixels back to real Vienna coordinates.
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print(f"Saved Vienna orthophoto to: {image_path}")
print(f"Saved georeferencing metadata to: {metadata_path}")

In [ ]:
print("Georeferencing metadata:")
print(json.dumps({
    "bbox": metadata["bbox"],
    "bbox_lonlat": metadata["bbox_lonlat"],
    "crs": metadata["crs"],
    "image_width": metadata["image_width"],
    "image_height": metadata["image_height"],
    "tile_matrix_set": metadata["tile_matrix_set"],
    "tile_matrix": metadata["tile_matrix"],
    "zoom": metadata["zoom"],
}, indent=2))

In [ ]:
# Display the orthophoto. The image is already georeferenced by the metadata above;
# the PNG itself is just the visual crop used by later detection experiments.
orthofoto = Image.open(image_path)
display(orthofoto)

plt.figure(figsize=(8, 8))
plt.imshow(orthofoto)
plt.axis("off")
plt.title("City of Vienna Orthofoto / ViennaGIS")
plt.show()

In [ ]:
# Quick coordinate checks for the helper functions.
x, y = lonlat_to_webmercator(lon, lat)
roundtrip_lon, roundtrip_lat = webmercator_to_lonlat(x, y)
center_lon, center_lat = image_pixel_to_lonlat(
    image_width / 2,
    image_height / 2,
    (metadata["bbox"]["min_x"], metadata["bbox"]["min_y"], metadata["bbox"]["max_x"], metadata["bbox"]["max_y"]),
    metadata["image_width"],
    metadata["image_height"],
)

print(f"Input lon/lat:      {lon:.6f}, {lat:.6f}")
print(f"Web Mercator:       {x:.2f}, {y:.2f}")
print(f"Roundtrip lon/lat:  {roundtrip_lon:.6f}, {roundtrip_lat:.6f}")
print(f"Center pixel lon/lat: {center_lon:.6f}, {center_lat:.6f}")